# 10 — Marketing ROI & Discount Effectiveness
Which discounts drive volume without destroying margin? Segment × discount tier matrix and sub-category ROI.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Segment × Discount Tier Margin Heatmap ────────────────────────────────────
matrix_data = fetch_df(cur, """
    SELECT c.segment,
           CASE
               WHEN oi.discount = 0    THEN 'No Discount'
               WHEN oi.discount <= 0.2 THEN 'Low (1-20%)'
               WHEN oi.discount <= 0.4 THEN 'Mid (21-40%)'
               ELSE 'High (>40%)'
           END AS discount_tier,
           COUNT(*)                        AS line_items,
           ROUND(SUM(oi.sales),2)          AS revenue,
           ROUND(SUM(oi.profit),2)         AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.segment,
        CASE
            WHEN oi.discount = 0    THEN 'No Discount'
            WHEN oi.discount <= 0.2 THEN 'Low (1-20%)'
            WHEN oi.discount <= 0.4 THEN 'Mid (21-40%)'
            ELSE 'High (>40%)'
        END
""")

tier_order = ["No Discount", "Low (1-20%)", "Mid (21-40%)", "High (>40%)"]
pivot_m    = matrix_data.pivot(index="segment", columns="discount_tier",
                               values="margin_pct")
pivot_m    = pivot_m.reindex(columns=tier_order)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot_m.values, cmap="RdYlGn", vmin=-20, vmax=40,
               aspect="auto")
ax.set_xticks(range(len(pivot_m.columns)))
ax.set_xticklabels(pivot_m.columns, rotation=15, fontsize=10)
ax.set_yticks(range(len(pivot_m.index)))
ax.set_yticklabels(pivot_m.index, fontsize=10)

for i in range(len(pivot_m.index)):
    for j in range(len(pivot_m.columns)):
        val = pivot_m.values[i, j]
        txt = f"{val:.1f}%" if not (val != val) else "N/A"
        ax.text(j, i, txt, ha="center", va="center",
                fontsize=12, fontweight="bold",
                color="white" if abs(val) > 15 else "black")

plt.colorbar(im, ax=ax, label="Profit Margin %", orientation="vertical",
             fraction=0.04)
ax.set_title("Profit Margin % — Segment × Discount Tier")
plt.tight_layout()
plt.show()


In [ ]:
# ── Sub-Category ROI % (Profit / Revenue) ─────────────────────────────────────
roi = fetch_df(cur, """
    SELECT p.sub_category,
           ROUND(SUM(oi.sales),2)  AS revenue,
           ROUND(SUM(oi.profit),2) AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS roi_pct
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.sub_category
    ORDER BY roi_pct DESC
""")

colors = [PALETTE[1] if r >= 0 else PALETTE[3] for r in roi["roi_pct"][::-1]]
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(roi["sub_category"][::-1], roi["roi_pct"][::-1],
               color=colors, edgecolor="white")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")

for bar in bars:
    w       = bar.get_width()
    offset  = 0.4 if w >= 0 else -0.4
    align   = "left" if w >= 0 else "right"
    ax.text(w + offset, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}%", va="center", ha=align, fontsize=9, fontweight="bold")

ax.set_xlabel("ROI %")
ax.set_title("Sub-Category ROI %  (Profit / Revenue × 100)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
